# Lab 12: Optimal Feedback Control 
**Computational Sensorimotor Control**

This lab builds the LQG controller for reaching:
1. **Linearize** the 2-link arm at the target posture
2. **LQR** — implement the backward Riccati recursion from scratch
3. **Simulate LQR** — deterministic reaching with full state feedback
4. **Kalman filter** — adapt the Week 9 KF for hand-state estimation
5. **LQG** — combine LQR + KF for reaching under noise
6. **Minimum intervention** — point target vs bar target
7. **OFC vs EPH** — compare with the Week 11 muscle model

We use a torque-driven arm (no muscles) to keep the math tractable.
Week 13 extends this to the full nonlinear Hill-type plant via iLQG.

---
## Part 0: Setup

In [ ]:
# Install the smc package (run once)
!pip install --break-system-packages -q git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.figsize': (10, 5), 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.family': 'serif'
})

from smc import Arm, Q_REF, mass_matrix
from smc import lambda_for_posture, make_ramp, simulate_lambda

arm = Arm()
fk = arm.forward_kinematics
ik = arm.inverse_kinematics

# Color palette (matches lecture figures)
C_MJ   = '#27AE60'   # min-jerk (green)
C_LQR  = '#2E86AB'   # LQR/proprioception (blue)
C_LQG  = '#E74C3C'   # LQG/vision (red)
C_EPH  = '#8E44AD'   # EPH (purple)
C_FOV  = '#F39C12'   # foveal (orange)

# ── Reach setup ──
start_hand = fk(Q_REF)
target_xy  = start_hand + np.array([0.12, 0.0])  # 12 cm rightward
q_target   = ik(target_xy[0], target_xy[1])

dt = 0.01   # 10 ms timestep
T  = 0.5    # 500 ms movement
N  = int(T / dt)

# Initial state: deviation from target in joint space
x0 = np.zeros(4)
x0[:2] = Q_REF - q_target

print(f'Start hand: ({start_hand[0]*100:.1f}, {start_hand[1]*100:.1f}) cm')
print(f'Target:     ({target_xy[0]*100:.1f}, {target_xy[1]*100:.1f}) cm')
print(f'Reach distance: {np.linalg.norm(target_xy - start_hand)*100:.1f} cm')
print(f'x0 = [{np.degrees(x0[0]):.1f}°, {np.degrees(x0[1]):.1f}°] joint deviation')

---
## Part 1: Linearize the Arm

Linearize the 2-link arm dynamics at the target posture q_target.

**State:** x = [δq₁, δq₂, δq̇₁, δq̇₂] (4×1) — deviations from target  
**Input:** u = [τ₁, τ₂] (2×1) — joint torques  
**Dynamics:** x_{t+1} = Ax_t + Bu_t

The continuous-time matrices are:
- A_c = [[0, I], [0, 0]] — drift (arm coasts)
- B_c = [[0], [M⁻¹]] — torque effect through inverse mass matrix

Discretize: A = I + A_c·dt, B = B_c·dt

*This reproduces Lecture §2.*

In [ ]:
def linearize_arm(q_lin, dt):
    """Linearize 2-link arm at (q_lin, 0). Returns A (4×4), B (4×2)."""
    # Continuous-time matrices:
    #   A_c = [[0 I], [0 0]]   (4×4)
    #   B_c = [[0], [M⁻¹]]    (4×2)
    # Discretize: A = I + A_c·dt,  B = B_c·dt
    
    Ac = np.zeros((4, 4))
    Ac[0, 2] = 1.0   # δq̇₁ drives δq₁
    Ac[1, 3] = 1.0   # δq̇₂ drives δq₂
    
    Bc = np.zeros((4, 2))
    # ════════════════════════════════════════════════════════════════
    # ▶ FILL IN (2 lines): Compute M⁻¹ and place it in Bc
    #   Hint: use mass_matrix(q_lin) and np.linalg.inv()
    # ════════════════════════════════════════════════════════════════
    ...  # YOUR CODE HERE
    ...  # YOUR CODE HERE
    # ════════════════════════════════════════════════════════════════
    
    A = np.eye(4) + Ac * dt
    B = Bc * dt
    return A, B

A, B = linearize_arm(q_target, dt)
print('A (4×4 state transition):'); print(np.round(A, 4))
print('\nB (4×2 input matrix):'); print(np.round(B, 6))

In [ ]:
def task_C(q_lin):
    """Task-space output: δp = J·δq → C = [J  0₂ₓ₂] maps 4D state to 2D hand position."""
    J = arm.jacobian(q_lin)
    C = np.zeros((2, 4))
    C[:, :2] = J
    return C

def task_C_vel(q_lin):
    """Full output for pos+vel: [J 0; 0 J] maps 4D state to 4D hand state."""
    J = arm.jacobian(q_lin)
    Cv = np.zeros((4, 4))
    Cv[:2, :2] = J
    Cv[2:, 2:] = J
    return Cv

C_out = task_C(q_target)
print('C (2×4 task-space output):'); print(np.round(C_out, 4))

---
## Part 2: Build the Cost Function and Solve LQR

The cost function J = Σ xᵀQ_t x + Σ uᵀRu penalizes state errors and torque magnitude.

**Your task:** Implement the backward Riccati recursion:
- S_N = Q_N (terminal cost)
- L_t = (R + BᵀS_{t+1}B)⁻¹ BᵀS_{t+1}A (optimal gain)
- S_t = Q_t + AᵀS_{t+1}(A − BL_t) (cost-to-go)

*This reproduces Lecture Figure 2.*

In [ ]:
def build_Q_sequence(q_lin, N, w_pos=100.0, w_vel=1.0, endpoint_only=True):
    """Build state-cost sequence Q_0 ... Q_N.
    
    endpoint_only=True:  Q_t = 0 for t < N, Q_N = w_pos·CᵀC + w_vel·I_vel
    endpoint_only=False: Q ramps up over last 30% + large terminal Q_N
    """
    C = task_C(q_lin)
    Q_pos = w_pos * (C.T @ C)   # task-space position cost (4×4)
    Q_vel = np.zeros((4, 4))
    Q_vel[2, 2] = w_vel
    Q_vel[3, 3] = w_vel
    Q_seq = []
    for t in range(N + 1):
        Q = np.zeros((4, 4))
        if endpoint_only:
            if t == N:
                Q = Q_pos + Q_vel
        else:
            frac = t / N
            if frac >= 0.7:
                s = ((frac - 0.7) / 0.3) ** 2
                Q = s * Q_pos + s * Q_vel
            if t == N:
                Q += 10 * Q_pos + 10 * Q_vel
        Q_seq.append(Q)
    return Q_seq

r = 0.0001
R_mat = r * np.eye(2)
print(f'R = {r} · I₂ (control cost)')

In [ ]:
def riccati_backward(A, B, Q_seq, R, N):
    """Backward Riccati recursion.
    Returns L (list of N gain matrices), S (list of N+1 cost-to-go matrices).
    """
    S = [None] * (N + 1)
    L = [None] * N
    # ════════════════════════════════════════════════════════════════
    # ▶ FILL IN (3 lines):
    #   1. Set terminal condition: S[N] = ?
    #   2. Inside the loop: compute L[t] from S[t+1]
    #   3. Inside the loop: compute S[t] from Q_seq[t], S[t+1], L[t]
    # ════════════════════════════════════════════════════════════════
    ...  # YOUR CODE HERE
    for t in range(N - 1, -1, -1):
        BtS = B.T @ S[t + 1]
        ...  # YOUR CODE HERE
        ...  # YOUR CODE HERE
    # ════════════════════════════════════════════════════════════════
    return L, S

# Solve for both cost configurations
Q_ep   = build_Q_sequence(q_target, N, endpoint_only=True)
Q_ramp = build_Q_sequence(q_target, N, endpoint_only=False)
L_ep,   S_ep   = riccati_backward(A, B, Q_ep,   R_mat, N)
L_ramp, S_ramp = riccati_backward(A, B, Q_ramp, R_mat, N)

print(f'L_ep[0]  shape: {L_ep[0].shape}  (2 torques × 4 state variables)')
print(f'‖L_ep[0]‖  = {np.linalg.norm(L_ep[0]):.1f}  (early: small)')
print(f'‖L_ep[-2]‖ = {np.linalg.norm(L_ep[-2]):.1f}  (late: large)')

In [ ]:
# ── Figure: LQR gains (reproduces Lecture Figure 2) ──
t_ms  = np.arange(N) * dt * 1000
tf_ms = np.arange(N + 1) * dt * 1000

gn_ep   = [np.linalg.norm(L_ep[t])   for t in range(N)]
gn_ramp = [np.linalg.norm(L_ramp[t]) for t in range(N)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(t_ms, gn_ep,   color=C_LQR, lw=2, label='Endpoint-only Q')
axes[0].plot(t_ms, gn_ramp, color=C_LQG, lw=2, label='Ramped Q')
axes[0].set_xlabel('Time (ms)'); axes[0].set_ylabel('‖L_t‖')
axes[0].set_title('A: LQR gain magnitude', fontweight='bold')
axes[0].legend()

labs = ['δq₁→τ₁', 'δq₂→τ₁', 'δq̇₁→τ₁', 'δq̇₂→τ₁']
cols = [C_LQR, C_LQG, C_MJ, C_FOV]
for j, (lab, col) in enumerate(zip(labs, cols)):
    axes[1].plot(t_ms, [L_ep[t][0, j] for t in range(N)], color=col, lw=2, label=lab)
axes[1].set_xlabel('Time (ms)'); axes[1].set_ylabel('Gain')
axes[1].set_title('B: Gain components (shoulder torque)', fontweight='bold')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

---
## Part 3: Simulate LQR (Deterministic)

Apply the optimal gains with full state feedback and no noise.
The control law is u_t = −L_t · x_t (true state, not estimated).

**Your task:** Implement the forward simulation loop.

*This reproduces Lecture Figure 3.*

In [ ]:
def simulate_lqr(A, B, L, x0, N):
    """Deterministic LQR: x_{t+1} = Ax + Bu, u = −L_t · x."""
    x = x0.copy()
    x_hist = np.zeros((N + 1, 4))
    u_hist = np.zeros((N, 2))
    x_hist[0] = x
    for t in range(N):
        # ════════════════════════════════════════════════════════════════
        # ▶ FILL IN: Compute optimal control and apply dynamics
        # ════════════════════════════════════════════════════════════════
        ...  # YOUR CODE HERE
        ...  # YOUR CODE HERE
        # ════════════════════════════════════════════════════════════════
        u_hist[t] = u
        x_hist[t + 1] = x
    return x_hist, u_hist

def states_to_hand(x_seq, q_ref):
    """Convert joint-space state sequence to hand positions in cm."""
    return np.array([fk(q_ref + x_seq[i, :2]) * 100 for i in range(len(x_seq))])

def min_jerk(start, end, N):
    """Min-jerk trajectory (5th order polynomial)."""
    tau = np.linspace(0, 1, N + 1)
    s = 10*tau**3 - 15*tau**4 + 6*tau**5
    return np.outer(1 - s, start) + np.outer(s, end)

In [ ]:
x_ep, _  = simulate_lqr(A, B, L_ep,   x0, N)
x_rmp, _ = simulate_lqr(A, B, L_ramp, x0, N)

h_ep  = states_to_hand(x_ep,  q_target)
h_rmp = states_to_hand(x_rmp, q_target)
mj    = min_jerk(start_hand * 100, target_xy * 100, N)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
ax = axes[0]
ax.plot(h_ep[:, 0], h_ep[:, 1], color=C_LQR, lw=2, label='Endpoint Q')
ax.plot(h_rmp[:, 0], h_rmp[:, 1], color=C_LQG, lw=2, label='Ramped Q')
ax.plot(mj[:, 0], mj[:, 1], '--', color=C_MJ, lw=2, label='Min-jerk')
ax.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8)
ax.plot(target_xy[0]*100, target_xy[1]*100, 'kx', ms=10, mew=2)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('A: Hand paths'); ax.legend(fontsize=9)

ax = axes[1]
for h, c, lab in [(h_ep, C_LQR, 'Endpoint'), (h_rmp, C_LQG, 'Ramped'), (mj, C_MJ, 'Min-jerk')]:
    d = np.linalg.norm(h - start_hand*100, axis=1)
    ax.plot(tf_ms, d, color=c, lw=2, ls='--' if lab=='Min-jerk' else '-', label=lab)
ax.axhline(12, color='gray', ls=':', lw=1)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Displacement (cm)')
ax.set_title('B: Displacement'); ax.legend(fontsize=9)

ax = axes[2]
for h, c, lab in [(h_ep, C_LQR, 'Endpoint'), (h_rmp, C_LQG, 'Ramped'), (mj, C_MJ, 'Min-jerk')]:
    v = np.linalg.norm(np.diff(h, axis=0), axis=1) / dt
    ax.plot(t_ms, v, color=c, lw=2, ls='--' if lab=='Min-jerk' else '-', label=lab)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Speed (cm/s)')
ax.set_title('C: Velocity profiles'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

---
## Part 4: Kalman Filter for Hand-State Estimation

Build the observation model (proprioception + vision) and implement the
KF predict-update cycle. The structure is the same as Week 9, but now
estimating the **hand state** instead of the target.

**Observation model:** y_t = H x_t + v_t, where H (8×4) stacks:
- Top 4 rows: I₄ (proprioception — measures joint state directly)
- Bottom 4 rows: C_vel (vision — measures hand pos+vel via Jacobian)

**Your task:** Implement `kalman_step()` with predict and update.

*This reproduces Lecture Figure 4.*

In [ ]:
def build_obs_model(q_lin):
    """Build H (8×4), V_periph (8×8), V_foveal (8×8)."""
    Cv = task_C_vel(q_lin)
    H = np.zeros((8, 4))
    H[:4, :] = np.eye(4)       # proprioception: measures joint state directly
    H[4:, :] = Cv               # vision: measures hand pos+vel via Jacobian
    
    V_periph = np.diag([
        np.radians(1.0)**2, np.radians(1.0)**2,   # proprioception position (σ=1°)
        np.radians(5.0)**2, np.radians(5.0)**2,   # proprioception velocity (σ=5°/s)
        0.005**2, 0.005**2,                        # peripheral vision position (σ=5mm)
        0.01**2,  0.01**2,                         # peripheral vision velocity (σ=1cm/s)
    ])
    V_foveal = np.diag([
        np.radians(1.0)**2, np.radians(1.0)**2,   # proprioception unchanged
        np.radians(5.0)**2, np.radians(5.0)**2,
        0.001**2, 0.001**2,                        # foveal vision position (σ=1mm)
        0.005**2, 0.005**2,                        # foveal vision velocity (σ=5mm/s)
    ])
    return H, V_periph, V_foveal

def sensory_schedule(N, transition_frac=0.70, width=0.08):
    """Sigmoid blend α_t: 0=peripheral, 1=foveal. Transition at ~70% of movement."""
    return np.array([1.0/(1.0+np.exp(-(t/N - transition_frac)/width)) for t in range(N)])

H_obs, V_per, V_fov = build_obs_model(q_target)
alpha = sensory_schedule(N)

# Process noise
W = np.diag([1e-6, 1e-6, 5e-4, 5e-4])

print(f'H shape: {H_obs.shape} (8 measurements × 4 state variables)')
print(f'Proprioception σ: pos={np.degrees(np.sqrt(V_per[0,0])):.1f}°, vel={np.degrees(np.sqrt(V_per[2,2])):.1f}°/s')
print(f'Peripheral vis σ: pos={np.sqrt(V_per[4,4])*1000:.0f}mm, vel={np.sqrt(V_per[6,6])*100:.0f}cm/s')
print(f'Foveal vis σ:     pos={np.sqrt(V_fov[4,4])*1000:.0f}mm, vel={np.sqrt(V_fov[6,6])*1000:.0f}mm/s')

In [ ]:
def kalman_step(x_hat, P, u, y, A, B, H, W, V):
    """Single KF predict-update cycle.
    Returns x_hat_new (4×1), P_new (4×4), K (4×8).
    """
    n = A.shape[0]
    
    # ── Predict (provided) ──
    x_pred = A @ x_hat + B @ u       # forward model + efference copy (0 ms delay)
    P_pred = A @ P @ A.T + W         # uncertainty grows
    
    # ════════════════════════════════════════════════════════════════
    # ▶ FILL IN (3 lines): Compute Kalman gain K, updated estimate,
    #   and updated covariance. Same structure as Week 9.
    #   Hint: innovation covariance S = H P_pred Hᵀ + V
    # ════════════════════════════════════════════════════════════════
    ...  # YOUR CODE HERE
    ...  # YOUR CODE HERE
    ...  # YOUR CODE HERE
    # ════════════════════════════════════════════════════════════════
    return x_new, P_new, K

---
## Part 5: Simulate LQG (200 Trials)

Combine LQR gains + Kalman filter. The control law is u_t = −L_t · x̂_t
(estimated state, not true state). This is the full LQG controller.

*This reproduces Lecture Figures 5 and 6.*

In [ ]:
def simulate_lqg(A, B, H, L, W, V_per, V_fov, alpha, x0, N,
                 rng=None, n_trials=1):
    """LQG simulation: LQR + Kalman filter with time-varying sensory noise."""
    n, m, p = 4, 2, H.shape[0]
    if rng is None: rng = np.random.default_rng(42)
    Wc = np.linalg.cholesky(W + 1e-14 * np.eye(n))
    x_all  = np.zeros((n_trials, N+1, n))
    xh_all = np.zeros((n_trials, N+1, n))
    u_all  = np.zeros((n_trials, N, m))
    K_store = np.zeros((N, n, p))
    for tr in range(n_trials):
        x  = x0.copy()
        xh = np.zeros(n)
        P  = 0.05 * np.eye(n)
        x_all[tr, 0] = x;  xh_all[tr, 0] = xh
        for t in range(N):
            u = -L[t] @ xh
            u_all[tr, t] = u
            x = A @ x + B @ u + Wc @ rng.standard_normal(n)
            x_all[tr, t+1] = x
            a = alpha[t]
            V_t = (1-a)*V_per + a*V_fov
            Vc = np.linalg.cholesky(V_t + 1e-14*np.eye(p))
            y = H @ x + Vc @ rng.standard_normal(p)
            xh, P, K = kalman_step(xh, P, u, y, A, B, H, W, V_t)
            xh_all[tr, t+1] = xh
            if tr == 0: K_store[t] = K
    return x_all, xh_all, u_all, K_store

n_trials = 200
x_lqg, xh_lqg, u_lqg, K_store = simulate_lqg(
    A, B, H_obs, L_ep, W, V_per, V_fov, alpha, x0, N,
    rng=np.random.default_rng(42), n_trials=n_trials)

# Convert to hand space
all_hands = np.zeros((n_trials, N+1, 2))
for tr in range(n_trials):
    all_hands[tr] = states_to_hand(x_lqg[tr], q_target)
h_mean = np.mean(all_hands, axis=0)
print(f'Mean endpoint: ({h_mean[-1, 0]:.2f}, {h_mean[-1, 1]:.2f}) cm')
print(f'Mean endpoint error: {np.linalg.norm(h_mean[-1] - target_xy*100):.2f} cm')

In [ ]:
# ── Figure: Kalman filter behavior (reproduces Lecture Figure 4) ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel A: Position Kalman gains by source
ax = axes[0]
K_prop_pos = [np.linalg.norm(K_store[t, :2, :4]) for t in range(N)]
K_vis_pos  = [np.linalg.norm(K_store[t, :2, 4:]) for t in range(N)]
ax.plot(t_ms, K_prop_pos, color=C_LQR, lw=2, label='Proprioception')
ax.plot(t_ms, K_vis_pos,  color=C_LQG, lw=2, label='Vision')
ax.axvline(350, color='gray', ls=':', lw=1)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('‖K‖ (position block)')
ax.set_title('A: Position estimate gains'); ax.legend(fontsize=9)

# Panel B: Velocity Kalman gains by source
ax = axes[1]
K_prop_vel = [np.linalg.norm(K_store[t, 2:4, :4]) for t in range(N)]
K_vis_vel  = [np.linalg.norm(K_store[t, 2:4, 4:]) for t in range(N)]
ax.plot(t_ms, K_prop_vel, color=C_LQR, lw=2, label='Proprioception')
ax.plot(t_ms, K_vis_vel,  color=C_LQG, lw=2, label='Vision')
ax.axvline(350, color='gray', ls=':', lw=1)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('‖K‖ (velocity block)')
ax.set_title('B: Velocity estimate gains'); ax.legend(fontsize=9)

# Panel C: Sensory transition + uncertainty
ax = axes[2]
ax.plot(t_ms, alpha, color=C_FOV, lw=2, label='α (foveal blend)')
ax.fill_between(t_ms, 0, alpha, color=C_FOV, alpha=0.15)
ax2 = ax.twinx()
Wc_ = np.linalg.cholesky(W + 1e-14*np.eye(4))
xh_ = np.zeros(4); P_ = 0.05 * np.eye(4); x_ = x0.copy()
rng_ = np.random.default_rng(99)
P_trace = np.zeros(N)
for t in range(N):
    u_ = -L_ep[t] @ xh_
    x_ = A @ x_ + B @ u_ + Wc_ @ rng_.standard_normal(4)
    V_t = (1-alpha[t])*V_per + alpha[t]*V_fov
    Vc_ = np.linalg.cholesky(V_t + 1e-14*np.eye(8))
    y_ = H_obs @ x_ + Vc_ @ rng_.standard_normal(8)
    xh_, P_, _ = kalman_step(xh_, P_, u_, y_, A, B, H_obs, W, V_t)
    P_trace[t] = np.trace(P_)
ax2.plot(t_ms, P_trace, color=C_EPH, lw=2, ls='--', label='tr(P)')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('α', color=C_FOV)
ax2.set_ylabel('tr(P)', color=C_EPH)
ax.set_title('C: Sensory transition & uncertainty')
li1, la1 = ax.get_legend_handles_labels()
li2, la2 = ax2.get_legend_handles_labels()
ax.legend(li1+li2, la1+la2, fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── Figure: Two gains, two directions (reproduces Lecture Figure 5) ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel A: Riccati gains (controller — backward pass)
ax = axes[0]
L_norm = [np.linalg.norm(L_ep[t]) for t in range(N)]
ax.plot(t_ms, L_norm, color=C_LQR, lw=2.5)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('‖L_t‖')
ax.set_title('A: Controller gains L_t (backward Riccati)')
ax.annotate('Gains peak\nnear endpoint', xy=(470, max(L_norm)*0.95), fontsize=9,
            xytext=(300, max(L_norm)*0.6), arrowprops=dict(arrowstyle='->', color='gray'))

# Panel B: Kalman gains (estimator — forward pass)
ax = axes[1]
K_total = [np.linalg.norm(K_store[t]) for t in range(N)]
ax.plot(t_ms, K_total, color=C_LQG, lw=2.5)
ax.axvline(350, color='gray', ls=':', lw=1)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('‖K_t‖ (Frobenius norm of 4×8 matrix)')
ax.set_title('B: Estimator gains K_t (forward Kalman)')
ax.annotate('Hand enters\nfoveal range', xy=(370, K_total[35]), fontsize=9,
            xytext=(200, max(K_total)*0.5), arrowprops=dict(arrowstyle='->', color='gray'))
plt.tight_layout(); plt.show()

print('L_t computed BEFORE movement (backward from endpoint). Shape: 2×4 per timestep.')
print('K_t computed DURING movement (forward with sensory data). Shape: 4×8 per timestep.')
print('Both must converge near the endpoint for the corrective phase to work.')

In [ ]:
# ── Figure: LQG reaching (reproduces Lecture Figure 6) ──
from matplotlib.patches import Ellipse
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

ax = axes[0]
for tr in range(min(n_trials, 30)):
    ax.plot(all_hands[tr, :, 0], all_hands[tr, :, 1], color=C_LQR, alpha=0.12, lw=0.8)
ax.plot(h_mean[:, 0], h_mean[:, 1], color=C_LQR, lw=2.5, label='Mean')
ax.plot(mj[:, 0], mj[:, 1], '--', color=C_MJ, lw=2, label='Min-jerk')
ax.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8)
ax.plot(target_xy[0]*100, target_xy[1]*100, 'kx', ms=10, mew=2)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('A: LQG hand paths (200 trials)'); ax.legend(fontsize=9)

ax = axes[1]
eps = all_hands[:, -1, :]
ax.scatter(eps[:, 0], eps[:, 1], c=C_LQR, alpha=0.5, s=25)
ax.plot(target_xy[0]*100, target_xy[1]*100, 'kx', ms=12, mew=2)
cov = np.cov(eps.T)
eigv, eigvec = np.linalg.eigh(cov)
ang = np.degrees(np.arctan2(eigvec[1,1], eigvec[0,1]))
ell = Ellipse(xy=np.mean(eps, axis=0), width=4*np.sqrt(max(eigv[1],1e-10)),
              height=4*np.sqrt(max(eigv[0],1e-10)), angle=ang, fc='none', ec=C_LQR, lw=2)
ax.add_patch(ell)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('B: Endpoint distribution (±2 SD)'); ax.set_aspect('equal')

ax = axes[2]
# Single trial: true state vs KF estimate
h_true = states_to_hand(x_lqg[0], q_target)
h_est  = states_to_hand(xh_lqg[0], q_target)
d_true = np.linalg.norm(h_true - start_hand*100, axis=1)
d_est  = np.linalg.norm(h_est  - start_hand*100, axis=1)
ax.plot(tf_ms, d_true, color=C_LQR, lw=2, label='True state')
ax.plot(tf_ms, d_est, '--', color=C_LQG, lw=2, label='KF estimate')
ax.axhline(12, color='gray', ls=':', lw=1)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Displacement (cm)')
ax.set_title('C: True state vs KF estimate'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

---
## Part 6: Minimum Intervention — Point vs Bar Target

The minimum intervention principle predicts that the controller only suppresses
variability in **task-relevant** dimensions. We test this by comparing two tasks:

- **Point target:** Q_N penalizes both x and y errors (both directions matter)
- **Bar target:** Q_N penalizes x strongly, y weakly (5 cm vertical bar — y is free within ±2.5 cm)

Same arm, same noise, same reach. Only Q changes.

**Your task:** Build the bar target Q_N matrix.

*This reproduces Lecture Figure 7.*

In [ ]:
# Point target: standard endpoint-only Q (penalizes both x and y)
Q_point = build_Q_sequence(q_target, N, w_pos=100.0, w_vel=1.0, endpoint_only=True)
L_point, _ = riccati_backward(A, B, Q_point, R_mat, N)

# Bar target: penalize x strongly, y weakly (5 cm vertical bar)
C_x = C_out[0:1, :]   # maps state → hand x-position (1×4)
C_y = C_out[1:2, :]   # maps state → hand y-position (1×4)

Q_bar = []
for t in range(N + 1):
    Q = np.zeros((4, 4))
    if t == N:
        # ════════════════════════════════════════════════════════════════
        # ▶ FILL IN (2 lines): Build Q_N that penalizes x heavily (w=100)
        #   and y weakly (w=5). Add velocity penalty (w=1) on diagonal.
        #   Hint: C_x.T @ C_x penalizes only x; C_y.T @ C_y penalizes only y
        # ════════════════════════════════════════════════════════════════
        ...  # YOUR CODE HERE
        ...  # YOUR CODE HERE
        # ════════════════════════════════════════════════════════════════
    Q_bar.append(Q)

L_bar, _ = riccati_backward(A, B, Q_bar, R_mat, N)
print(f'Point target Q_N rank: {np.linalg.matrix_rank(Q_point[N])}')
print(f'Bar target Q_N rank:   {np.linalg.matrix_rank(Q_bar[N])}')

In [ ]:
# Simulate 50 trials per condition (same noise seed for fair comparison)
n_mip = 50
x_point, _, _, _ = simulate_lqg(A, B, H_obs, L_point, W, V_per, V_fov, alpha, x0, N,
    rng=np.random.default_rng(42), n_trials=n_mip)
x_bar, _, _, _ = simulate_lqg(A, B, H_obs, L_bar, W, V_per, V_fov, alpha, x0, N,
    rng=np.random.default_rng(42), n_trials=n_mip)

hands_point = np.zeros((n_mip, N+1, 2))
hands_bar   = np.zeros((n_mip, N+1, 2))
for tr in range(n_mip):
    hands_point[tr] = states_to_hand(x_point[tr], q_target)
    hands_bar[tr]   = states_to_hand(x_bar[tr], q_target)

# Reach direction for variance decomposition
reach_dir = (target_xy - start_hand); reach_dir /= np.linalg.norm(reach_dir)
perp_dir  = np.array([-reach_dir[1], reach_dir[0]])
bar_center = target_xy * 100

import matplotlib.lines as mlines
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Panel A: hand paths
ax = axes[0]
for tr in range(min(n_mip, 25)):
    ax.plot(hands_point[tr, :, 0], hands_point[tr, :, 1], color=C_LQR, alpha=0.12, lw=0.8)
    ax.plot(hands_bar[tr, :, 0], hands_bar[tr, :, 1], color=C_LQG, alpha=0.12, lw=0.8)
ax.plot([bar_center[0], bar_center[0]], [bar_center[1]-2.5, bar_center[1]+2.5],
        color=C_LQG, lw=4, alpha=0.4, solid_capstyle='round')
ax.plot(bar_center[0], bar_center[1], 'kx', ms=10, mew=2)
ax.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8)
ax.legend(handles=[
    mlines.Line2D([], [], color=C_LQR, lw=2, label='Point target'),
    mlines.Line2D([], [], color=C_LQG, lw=2, label='Bar target')], fontsize=9)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('A: Point vs bar target')
# Expand y-axis to show trajectory spread
all_y_mip = np.concatenate([hands_point[:, :, 1].ravel(), hands_bar[:, :, 1].ravel()])
y_mid_m = np.mean([np.min(all_y_mip), np.max(all_y_mip)])
y_span_m = max(np.max(all_y_mip) - np.min(all_y_mip), 2.0)
ax.set_ylim(y_mid_m - y_span_m * 0.7, y_mid_m + y_span_m * 0.7)

# Panel B: endpoint scatter
ax = axes[1]
ep_point = hands_point[:, -1, :]
ep_bar   = hands_bar[:, -1, :]
ax.scatter(ep_point[:, 0], ep_point[:, 1], c=C_LQR, alpha=0.5, s=25, label='Point')
ax.scatter(ep_bar[:, 0], ep_bar[:, 1], c=C_LQG, alpha=0.5, s=25, label='Bar')
ax.plot(bar_center[0], bar_center[1], 'kx', ms=12, mew=2)
ax.plot([bar_center[0], bar_center[0]], [bar_center[1]-2.5, bar_center[1]+2.5],
        color=C_LQG, lw=4, alpha=0.3, solid_capstyle='round')
for ep, color in [(ep_point, C_LQR), (ep_bar, C_LQG)]:
    cov_ep = np.cov(ep.T)
    eigv, eigvec = np.linalg.eigh(cov_ep)
    ang = np.degrees(np.arctan2(eigvec[1,1], eigvec[0,1]))
    ell = Ellipse(xy=np.mean(ep, axis=0), width=4*np.sqrt(max(eigv[1],1e-10)),
                  height=4*np.sqrt(max(eigv[0],1e-10)), angle=ang, fc='none', ec=color, lw=2)
    ax.add_patch(ell)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('B: Endpoint distributions (±2 SD)'); ax.legend(fontsize=9); ax.set_aspect('equal')

# Panel C: variance decomposition
ax = axes[2]
width = 0.35; x_pos = np.array([0, 1])
for i, (label, ep, color) in enumerate(zip(['Point', 'Bar'],
                                            [ep_point, ep_bar],
                                            [C_LQR, C_LQG])):
    ep_c = ep - np.mean(ep, axis=0)
    var_a = np.var(ep_c @ reach_dir)
    var_p = np.var(ep_c @ perp_dir)
    ax.bar(x_pos[i] - width/2, var_a, width, color=color, alpha=0.7)
    ax.bar(x_pos[i] + width/2, var_p, width, color=color, alpha=0.35)
    ax.text(x_pos[i] - width/2, var_a + 0.002, f'{var_a:.3f}', ha='center', va='bottom', fontsize=8)
    ax.text(x_pos[i] + width/2, var_p + 0.002, f'{var_p:.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x_pos); ax.set_xticklabels(['Point target', 'Bar target'])
ax.set_ylabel('Endpoint variance (cm²)')
ax.set_title('C: Variance decomposition')
ax.legend(handles=[
    mlines.Line2D([], [], color='gray', lw=8, alpha=0.7, label='Along reach'),
    mlines.Line2D([], [], color='gray', lw=8, alpha=0.35, label='Perpendicular')], fontsize=9)
plt.tight_layout(); plt.show()

print(f'\nPoint target: along={np.var((ep_point-np.mean(ep_point,0)) @ reach_dir):.3f}, '
      f'perp={np.var((ep_point-np.mean(ep_point,0)) @ perp_dir):.3f} cm²')
print(f'Bar target:   along={np.var((ep_bar-np.mean(ep_bar,0)) @ reach_dir):.3f}, '
      f'perp={np.var((ep_bar-np.mean(ep_bar,0)) @ perp_dir):.3f} cm²')

---
## Part 7: OFC vs EPH

Compare the LQG controller with the EPH (R-C Ca²⁺ λ) model from Week 4/11.
The EPH λ ramp aims at 10.2 cm (not 12 cm) because the Ca²⁺ model overshoots
by ~2%, settling at approximately 12 cm.

*This reproduces Lecture Figure 8.*

In [ ]:
# ── EPH model ──
li = lambda_for_posture(Q_REF)
# EPH aims at 10.2 cm — overshoot settles at ~12 cm
q_eph_aim = ik(*(start_hand + np.array([0.102, 0.0])))
lf = lambda_for_posture(q_eph_aim)
lam_fn = make_ramp(li, lf, t_start=0.05, duration=0.30)
t_eph, s_eph, h_eph, _ = simulate_lambda(lam_fn, T=0.9, dt=0.0001)
h_eph_cm = h_eph * 100

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Panel A: Hand paths
ax = axes[0]
ax.plot(h_mean[:, 0], h_mean[:, 1], color=C_LQR, lw=2.5, label='OFC (LQG)')
ax.plot(h_eph_cm[:, 0], h_eph_cm[:, 1], color=C_EPH, lw=2, label='EPH (R-C Ca²⁺ λ)')
ax.plot(mj[:, 0], mj[:, 1], '--', color=C_MJ, lw=2, label='Min-jerk')
ax.plot(start_hand[0]*100, start_hand[1]*100, 'ko', ms=8)
ax.plot(target_xy[0]*100, target_xy[1]*100, 'kx', ms=10, mew=2)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.set_title('A: Hand paths'); ax.legend(fontsize=9)

# Panel B: Displacement
ax = axes[1]
d_ofc = np.linalg.norm(h_mean - start_hand*100, axis=1)
d_eph = np.linalg.norm(h_eph_cm - start_hand*100, axis=1)
d_mj  = np.linalg.norm(mj - start_hand*100, axis=1)
ax.plot(tf_ms, d_ofc, color=C_LQR, lw=2, label='OFC')
ax.plot(t_eph*1000, d_eph, color=C_EPH, lw=2, label='EPH')
ax.plot(tf_ms, d_mj, '--', color=C_MJ, lw=2, label='Min-jerk')
ax.axhline(12, color='gray', ls=':', lw=1)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Displacement (cm)')
ax.set_title('B: Displacement'); ax.set_xlim(0, 850); ax.legend(fontsize=9)

# Panel C: Velocity profiles (vector-averaged for OFC)
ax = axes[2]
v_vec = np.diff(all_hands, axis=1) / dt  # (n_trials, N, 2)
v_ofc = np.linalg.norm(np.mean(v_vec, axis=0), axis=1)  # speed of mean velocity
v_eph = np.linalg.norm(np.diff(h_eph_cm, axis=0), axis=1) / 0.0001
v_mj  = np.linalg.norm(np.diff(mj, axis=0), axis=1) / dt
ax.plot(t_ms, v_ofc, color=C_LQR, lw=2, label='OFC')
ax.plot(t_eph[:-1]*1000, v_eph, color=C_EPH, lw=2, label='EPH')
ax.plot(t_ms, v_mj, '--', color=C_MJ, lw=2, label='Min-jerk')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Speed (cm/s)')
ax.set_title('C: Velocity profiles'); ax.set_xlim(0, 850); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()